## 1. 미니 배치와 배치 크기
- 전체 데이터에 대해서 한 번에 경사 하강법을 수행하는 방법을 '배치 경사 하강법'이라고 부릅니다. 반면, 미니 배치 단위로 경사 하강법을 수행하는 방법을 '미니 배치 경사 하강법'이라고 부릅니다.

- 배치 경사 하강법은 경사 하강법을 할 때, 전체 데이터를 사용하므로 가중치 값이 최적값에 수렴하는 과정이 매우 안정적이지만, 계산량이 너무 많이 듭니다. 미니 배치 경사 하강법은 경사 하강법을 할 때, 전체 데이터의 일부만을 보고 수행하므로 최적값으로 수렴하는 과정에서 값이 조금 헤매기도 하지만 훈련 속도가 빠릅니다.

- 배치 크기는 보통 2의 제곱수를 사용합니다. ex) 2, 4, 8, 16, 32, 64... 그 이유는 CPU와 GPU의 메모리가 2의 배수이므로 배치크기가 2의 제곱수일 경우에 데이터 송수신의 효율을 높일 수 있다고 합니다.

## 2. 이터레이션
- 이터레이션은 한 번의 에포크 내에서 이루어지는 매개변수인 가중치 `W`와 `b`의 업데이트 횟수입니다. 전체 데이터가 2,000일 때 배치 크기를 200으로 한다면 이터레이션의 수는 총 10개입니다. 이는 한 번의 에포크 당 매개변수 업데이트가 10번 이루어짐을 의미합니다.

## 3. 데이터 로드하기(Data Load)
- `TensorDataset`: 텐서를 입력받아 Dataset의 형태로 변환
- `DataLoader`: Dataset을 모델에 적재하며, 미니 배치의 크기, 셔플 여부 등을 지원

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
from torch.utils.data import TensorDataset # 텐서데이터셋
from torch.utils.data import DataLoader # 데이터로더

In [3]:
x_train  =  torch.FloatTensor([[73,  80,  75],
                               [93,  88,  93],
                               [89,  91,  90],
                               [96,  98,  100],
                               [73,  66,  70]])
y_train  =  torch.FloatTensor([[152],  [185],  [180],  [196],  [142]])

In [4]:
dataset = TensorDataset(x_train, y_train)

In [5]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [6]:
model = nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

In [7]:
nb_epochs = 20
for epoch in range(nb_epochs + 1):
  for batch_idx, samples in enumerate(dataloader):
    # print(batch_idx)
    # print(samples)
    x_train, y_train = samples
    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train)

    # cost로 H(x) 계산
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
        epoch, nb_epochs, batch_idx+1, len(dataloader),
        cost.item()
        ))

Epoch    0/20 Batch 1/3 Cost: 42913.117188
Epoch    0/20 Batch 2/3 Cost: 13487.091797
Epoch    0/20 Batch 3/3 Cost: 5322.784180
Epoch    1/20 Batch 1/3 Cost: 1152.333862
Epoch    1/20 Batch 2/3 Cost: 403.461487
Epoch    1/20 Batch 3/3 Cost: 130.981903
Epoch    2/20 Batch 1/3 Cost: 45.300884
Epoch    2/20 Batch 2/3 Cost: 7.250121
Epoch    2/20 Batch 3/3 Cost: 0.618965
Epoch    3/20 Batch 1/3 Cost: 0.560113
Epoch    3/20 Batch 2/3 Cost: 4.365398
Epoch    3/20 Batch 3/3 Cost: 0.012201
Epoch    4/20 Batch 1/3 Cost: 0.070687
Epoch    4/20 Batch 2/3 Cost: 2.478718
Epoch    4/20 Batch 3/3 Cost: 0.471629
Epoch    5/20 Batch 1/3 Cost: 2.239697
Epoch    5/20 Batch 2/3 Cost: 0.273194
Epoch    5/20 Batch 3/3 Cost: 0.120070
Epoch    6/20 Batch 1/3 Cost: 2.304333
Epoch    6/20 Batch 2/3 Cost: 0.348642
Epoch    6/20 Batch 3/3 Cost: 0.002706
Epoch    7/20 Batch 1/3 Cost: 0.110816
Epoch    7/20 Batch 2/3 Cost: 2.399700
Epoch    7/20 Batch 3/3 Cost: 0.077819
Epoch    8/20 Batch 1/3 Cost: 0.135815
Epoch 

In [8]:
# 임의의 입력 [73, 80, 75]를 선언
new_var =  torch.FloatTensor([[73, 80, 75]])
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var)
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[149.9547]], grad_fn=<AddmmBackward0>)


## 5. 커스텀 데이터셋(Custom Dataset)으로 선형 회귀 구현하기

In [9]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [10]:
# Dataset 상속
class CustomDataset(Dataset):
  def __init__(self, x, y):
    if len(x) % len(y) != 0:
      raise ValueError("x는 y로 나누어 떨어져야 합니다.")
    
    self.x_data = torch.FloatTensor(x).view(len(y), -1)
    self.y_data = torch.FloatTensor(y)

  # 총 데이터의 개수를 리턴
  def __len__(self):
    return len(self.x_data)

  # 인덱스를 입력받아 그에 맵핑되는 입출력 데이터를 파이토치의 Tensor 형태로 리턴
  def __getitem__(self, idx):
    x = torch.FloatTensor(self.x_data[idx])
    y = torch.FloatTensor(self.y_data[idx])
    return x, y

In [11]:
x = [73, 80, 75, 93, 88, 93, 89, 91, 90, 96, 98, 100, 73, 66, 70]
y = [152, 185, 180, 196, 142]

dataset = CustomDataset(x, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [12]:
model = torch.nn.Linear(3,1)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5)

In [13]:
nb_epochs = 20
for epoch in range(nb_epochs + 1):
  for batch_idx, samples in enumerate(dataloader):
    # print(batch_idx)
    # print(samples)
    x_train, y_train = samples
    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.mse_loss(prediction, y_train)

    # cost로 H(x) 계산
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} Batch {}/{} Cost: {:.6f}'.format(
        epoch, nb_epochs, batch_idx+1, len(dataloader),
        cost.item()
        ))

Epoch    0/20 Batch 1/3 Cost: 36999.816406
Epoch    0/20 Batch 2/3 Cost: 27982.003906
Epoch    0/20 Batch 3/3 Cost: 6211.940430
Epoch    1/20 Batch 1/3 Cost: 1607.957275
Epoch    1/20 Batch 2/3 Cost: 587.301331
Epoch    1/20 Batch 3/3 Cost: 116.963615
Epoch    2/20 Batch 1/3 Cost: 409.430145
Epoch    2/20 Batch 2/3 Cost: 1540.700317
Epoch    2/20 Batch 3/3 Cost: 39.609524
Epoch    3/20 Batch 1/3 Cost: 1547.608154
Epoch    3/20 Batch 2/3 Cost: 18.650265
Epoch    3/20 Batch 3/3 Cost: 0.147493
Epoch    4/20 Batch 1/3 Cost: 69.854469
Epoch    4/20 Batch 2/3 Cost: 8.957670
Epoch    4/20 Batch 3/3 Cost: 1.269267
Epoch    5/20 Batch 1/3 Cost: 77.861298
Epoch    5/20 Batch 2/3 Cost: 68.425781
Epoch    5/20 Batch 3/3 Cost: 0.513032
Epoch    6/20 Batch 1/3 Cost: 68.405220
Epoch    6/20 Batch 2/3 Cost: 8.307499
Epoch    6/20 Batch 3/3 Cost: 1.946037
Epoch    7/20 Batch 1/3 Cost: 391.876495
Epoch    7/20 Batch 2/3 Cost: 78.758202
Epoch    7/20 Batch 3/3 Cost: 7.794027
Epoch    8/20 Batch 1/3 Cost:

/tmp/ipykernel_41497/2514251569.py:11: UserWarning: Using a target size (torch.Size([2])) that is different to the input size (torch.Size([2, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  cost = F.mse_loss(prediction, y_train)
/tmp/ipykernel_41497/2514251569.py:11: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  cost = F.mse_loss(prediction, y_train)


In [14]:
# 임의의 입력 [73, 80, 75]를 선언
new_var =  torch.FloatTensor([[73, 80, 75]])
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var)
print("훈련 후 입력이 73, 80, 75일 때의 예측값 :", pred_y)

훈련 후 입력이 73, 80, 75일 때의 예측값 : tensor([[151.8051]], grad_fn=<AddmmBackward0>)
